#**An Exploratory Data Analysis (EDA) for the FMA-Small Dataset**
**Notebook by: Elia Fel P. Bado**

#**I. Introduction**

####**Data Source & Origin**
*   Repository Link: [FMA GitHub Repository](https://github.com/mdeff/fma)
*   Citation: Defferrard, M., Benzi, K., Vandergheynst, P., & Bresson, X. (2017). *FMA: A Dataset for Music Analysis*. arXiv:1612.01840.
<br>

####**Who collected the data, when, how, and for what purpose?**
* The dataset was created by Michaël Defferrard, Kirell Benzi, Pierre Vandergheynst, and Xavier Bresson in 2017 for the 18th International Society for Music Information Retrieval Conference (ISMIR). It was designed to provide an open, easily accessible audio dataset for evaluating tasks in Music Information Retrieval (MIR), aiming to overcome hurdles from the limited availability of large public audio datasets. The data was collected by querying the Free Music Archive (FMA) web API.
<br>

####**What does one row represent?**
* `tracks.csv`: One row represents a single music track.
* `features.csv`: One row represents extracted audio features (via librosa) for a track.
* `genres.csv`: One row represents a single musical genre.
* `echonest.csv`: One row represents Echonest-specific audio features for a track.
<br>

####**What does each column mean?**
* `tracks.csv`: Contains track metadata such as track ID, title, artist, genres, tags, and play counts.
* `features.csv`: Contains statistical descriptors of common audio features extracted using `librosa` (e.g., MFCCs, spectral centroid, chroma).
* `genres.csv`: Details 163 genres, including name, ID, and parent genre ID to infer hierarchy.
* `echonest.csv`: Provides Echonest audio features for a subset of 13,129 tracks.
* **NEED DATA DICTIONARY**

<br>


####**Questions to Explore**


1.   How accurately can baseline machine learning models perform genre recognition using extracted audio features vs. raw audio?
2.   How does the hierarchical genre taxonomy affect classification performance across top-level genres?
3.   Can representation learning on artist labels transfer effectively to genre classification tasks?
4.   How do Echonest audio features compare with librosa-extracted features in predicting track play counts?




In [22]:
import os

# connecting colab to gdrive, check if folder exists, and return folder size
from google.colab import drive
drive.mount('/content/drive')

#paths
base_path = '/content/drive/MyDrive/Music_Genre_Classification'
dataset_path = os.path.join(base_path, 'dataset')

raw_dataset_path = os.path.join(dataset_path, 'raw')
fma_metadata_path = os.path.join(raw_dataset_path, 'fma_metadata')
fma_small_path = os.path.join(raw_dataset_path, 'fma_small')

processed_dataset_path = os.path.join(dataset_path, 'processed')


#for fetching folder size
def get_dir_size(path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
    return total_size

#to check if folders has contents
folder_exists = os.path.exists(base_path)
print(f"Music_Genre_Classification file exists: {folder_exists}")
print(f"dataset/raw/fma_metadata exists: {os.path.exists(fma_metadata_path)}")
print(f"dataset/raw/fma_small exists: {os.path.exists(fma_small_path)}")

if folder_exists:
    fma_metadata_file_size = (get_dir_size(fma_metadata_path))/(1024*1024)
    fma_small_file_size = (get_dir_size(fma_small_path))/(1024*1024)
    print(f"fma_metadata Folder Size: {fma_metadata_file_size:.2f} MB")
    print(f"fma_small Folder Size: {fma_small_file_size:.2f} MB")
else:
    print("Folder Size: N/A")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Music_Genre_Classification file exists: True
dataset/raw/fma_metadata exists: True
dataset/raw/fma_small exists: True
fma_metadata Folder Size: 1396.98 MB
fma_small Folder Size: 7606.00 MB


# **II. Data loading and first look**

Only the `tracks.csv` and `features.csv` are the useful metadata for this reserach so these two will be the only ones to be inspected.

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [49]:
# DATA LOADING FOR tracks.csv

tracks_path = os.path.join(fma_metadata_path, 'tracks.csv')

# load the raw metadata handling the multi-index header
tracks = pd.read_csv(tracks_path, index_col=0, header=[0, 1])

# filtering the tracks to tee fma-small dataset
small_tracks = tracks[tracks[('set', 'subset')] == 'small']

#print the shape, sample, info, and description
display(Markdown("\n###--- **Tracks Shape ---**"))
print("Tracks Shape: ",small_tracks.shape)

display(Markdown("\n###--- **Tracks Random Sample (5 rows) ---**"))
display(small_tracks.sample(5, random_state=67))

display(Markdown("\n###--- **Tracks Info ---**"))
small_tracks.info()

display(Markdown("\n###--- **Tracks Description ---**"))
# Enable pandas to display all rows/columns so nothing gets truncated in the summary
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(small_tracks.describe(include='all').T)


###--- **Tracks Shape ---**

Tracks Shape:  (8000, 52)



###--- **Tracks Random Sample (5 rows) ---**

album                                                           \
         comments         date_created        date_released       engineer   
track_id                                                                     
60736           0  2012-02-16 12:21:38                  NaN            NaN   
35534           0  2010-09-13 21:34:29  2008-08-30 00:00:00  Mark Whitcomb   
131939          0  2016-02-05 17:24:58                  NaN            NaN   
71508           5  2012-10-19 17:36:21                  NaN            NaN   
76079           2  2013-01-14 16:39:08  2012-12-24 00:00:00            NaN   

                                                                              \
         favorites     id                                        information   
track_id                                                                       
60736            0  11111  <p><span style="color:#222222;font-family:Aria...   
35534            1   7319                                            <p></p>   
131939           0  19874  <p>Psychedelic electric Albanian, Macedonian, ...   
71508            3  12672  <p style="margin: 0px; padding: 8px 0px; font-...   
76079            0  13312  <p>The Koliadnyky are a group of Hutsuls from ...   

                                                                              \
         listens producer                                               tags   
track_id                                                                       
60736      20604      NaN  ['live', 'newyx', 'suicidal birds', 'recipe bo...   
35534      30930      NaN                                                 []   
131939      8794      NaN                                                 []   
71508     115471      NaN                                    ['peter swimm']   
76079      10119      NaN  ['trembita', 'ukraine', 'tsymbaly', 'koliadnyk...   

          ...       track                         \
          ... information interest language_code   
track_id  ...                                      
60736     ...         NaN     9012           NaN   
35534     ...         NaN     6328           NaN   
131939    ...         NaN     1041           NaN   
71508     ...         NaN     6193           NaN   
76079     ...         NaN     1105           NaN   

                                                                              \
                                                    license listens lyricist   
track_id                                                                       
60736     Attribution-Noncommercial-Share Alike 3.0 Unit...    7628      NaN   
35534              Attribution-ShareAlike 3.0 International    3553      NaN   
131939    Creative Commons Attribution-NonCommercial-NoD...     867      NaN   
71508                          Creative Commons Attribution    4609      NaN   
76079     Attribution-NonCommercial-NoDerivatives (aka M...     979      NaN   

                                                                              \
         number publisher                                               tags   
track_id                                                                       
60736        10       NaN  ['live', 'newyx', 'suicidal birds', 'recipe bo...   
35534        11       NaN                                                 []   
131939        1       NaN                                                 []   
71508         9       NaN                                    ['peter swimm']   
76079        13       NaN  ['trembita', 'ukraine', 'tsymbaly', 'koliadnyk...   

                                                             
                                                      title  
track_id                                                     
60736                   Mexican Holiday live at the Winston  
35534                                          Feed The One  
131939                                 Rreth e qark beratit  
71508                             


###--- **Tracks Info ---**

<class 'pandas.core.frame.DataFrame'>
Index: 8000 entries, 2 to 155066
Data columns (total 52 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   (album, comments)            8000 non-null   int64  
 1   (album, date_created)        8000 non-null   object 
 2   (album, date_released)       5337 non-null   object 
 3   (album, engineer)            1142 non-null   object 
 4   (album, favorites)           8000 non-null   int64  
 5   (album, id)                  8000 non-null   int64  
 6   (album, information)         6502 non-null   object 
 7   (album, listens)             8000 non-null   int64  
 8   (album, producer)            1404 non-null   object 
 9   (album, tags)                8000 non-null   object 
 10  (album, title)               8000 non-null   object 
 11  (album, tracks)              8000 non-null   int64  
 12  (album, type)                7769 non-null   object 
 13  (artist, active_year_


###--- **Tracks Description ---**

count unique  \
album  comments           8000.0    NaN   
       date_created         8000   2459   
       date_released        5337   1223   
       engineer             1142    145   
       favorites          8000.0    NaN   
       id                 8000.0    NaN   
       information          6502   1942   
       listens            8000.0    NaN   
       producer             1404    191   
       tags                 8000    378   
       title                8000   2417   
       tracks             8000.0    NaN   
       type                 7769      4   
artist active_year_begin    1542     49   
       active_year_end       358     32   
       associated_labels     849    180   
       bio                  5914   1357   
       comments           8000.0    NaN   
       date_created         8000   2301   
       favorites          8000.0    NaN   
       id                 8000.0    NaN   
       latitude           2899.0    NaN   
       location             5471    754   
       longitude          2899.0    NaN   
       members              3525    865   
       name                 8000   2306   
       related_projects      694    171   
       tags                 8000   2262   
       website              6452   1481   
       wikipedia_page        410    102   
set    split                8000      3   
       subset               8000      1   
track  bit_rate           8000.0    NaN   
       comments           8000.0    NaN   
       composer              180     68   
       date_created         8000   7000   
       date_recorded         465    116   
       duration           8000.0    NaN   
       favorites          8000.0    NaN   
       genre_top            8000      8   
       genres               8000    586   
       genres_all           8000    442   
       information           159    112   
       interest           8000.0    NaN   
       language_code        1005     13   
       license              7995     62   
       listens            8000.0    NaN   
       lyricist               26     12   
       number             8000.0    NaN   
       publisher              76     19   
       tags                 8000    378   
       title                8000   7767   

                                                                        top  \
album  comments                                                         NaN   
       date_created                                     2015-05-21 17:43:17   
       date_released                                    2007-01-01 00:00:00   
       engineer                                                Irene Trudel   
       favorites                                                        NaN   
       id                                                               NaN   
       information                    <p>Instrumental - Various styles.</p>   
       listens                                                          NaN   
       producer                                                Rob Weisberg   
       tags                                                              []   
       title                               Live at the 2014 Golden Festival   
       tracks                                                           NaN   
       type                                                           Album   
artist active_year_begin                                2008-01-01 00:00:00   
       active_year_end                                  2015-01-01 00:00:00   
       associated_labels  Time-Lag, Phase!, Feeding Tube, Peasant Magik,...   
       bio                <p><a title="Blue Dot Website" href="/contribu...   
       comments                                                         NaN   
       date_created                                     2015-02-03 12:05:36   
       favorites                                                        NaN   
       id                                                               NaN   
       la

In [50]:
# DATA LOADING FOR features.csv

features_path = os.path.join(fma_metadata_path, 'features.csv')

# load the raw metadata handling the multi-index header
features = pd.read_csv(features_path, index_col=0, header=[0, 1, 2])

#filter features to match the track_ids of the small subset
small_features = features.loc[small_tracks.index]

#print the shape, sample, and info
display(Markdown("\n###--- **Features Shape ---**"))
print("Features Shape: ",small_features.shape)

display(Markdown("\n###--- **Features Random Sample (5 rows) ---**"))
display(small_features.sample(5, random_state=67))

display(Markdown("\n###--- **Features Info ---**"))
small_features.info()

display(Markdown("\n###--- **Features Description ---**"))
# Enable pandas to display all rows/columns so nothing gets truncated in the summary
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(small_features.describe(include='all').T)


###--- **Features Shape ---**

Features Shape:  (8000, 518)



###--- **Features Random Sample (5 rows) ---**

feature    chroma_cens                                                    \
statistics    kurtosis                                                     
number              01        02        03        04        05        06   
track_id                                                                   
60736         3.666378  4.473220  4.830767  1.744571  1.397857  1.820356   
35534        -0.647455 -0.984537 -0.435043 -0.678039 -0.683269  0.023716   
131939       -1.180873 -0.374147 -1.011606 -0.172726 -0.229813  0.281381   
71508         1.639837 -1.019516  0.208297  3.664822  1.336885  0.702219   
76079        -1.134631 -0.795856 -1.140404 -1.008961  1.580629 -0.199513   

feature                                             ...   tonnetz            \
statistics                                          ...       std             
number            07        08        09        10  ...        04        05   
track_id                                            ...                       
60736       1.757500  4.938700  0.215622 -0.046953  ...  0.044820  0.015213   
35534      -0.257244 -0.170678  0.088068 -0.125568  ...  0.092387  0.019067   
131939     -0.905243  0.048887 -0.177934  0.298722  ...  0.121513  0.025258   
71508       3.188430  1.107264 -0.845742  0.570397  ...  0.068503  0.031183   
76079      -0.664471 -0.564692 -0.145973  0.189562  ...  0.111757  0.018231   

feature                     zcr                                          \
statistics             kurtosis       max      mean    median       min   
number            06         01        01        01        01        01   
track_id                                                                  
60736       0.011291   0.315925  0.107910  0.066835  0.067383  0.014160   
35534       0.018861  10.991987  0.349609  0.064351  0.056152  0.008789   
131939      0.022618   4.100006  0.149414  0.034082  0.030273  0.001953   
71508       0.014314   3.373695  0.228027  0.054972  0.052246  0.002441   
76079       0.019553   9.288609  0.406738  0.075960  0.060059  0.013184   

feature                         
statistics      skew       std  
number            01        01  
track_id                        
60736      -0.353675  0.012615  
35534       2.836525  0.037081  
131939      1.676785  0.015956  
71508       1.088970  0.025137  
76079       2.881972  0.056434  

[5 rows x 518 columns]


###--- **Features Info ---**

<class 'pandas.core.frame.DataFrame'>
Index: 8000 entries, 2 to 155066
Columns: 518 entries, ('chroma_cens', 'kurtosis', '01') to ('zcr', 'std', '01')
dtypes: float64(518)
memory usage: 31.7 MB



###--- **Features Description ---**

count         mean          std  \
feature            statistics number                                     
chroma_cens        kurtosis   01      8000.0     0.111983     1.656377   
                              02      8000.0     0.007180     1.934319   
                              03      8000.0     0.200494     6.073813   
                              04      8000.0     0.005261     1.288859   
                              05      8000.0     0.067485     1.636802   
                              06      8000.0     0.077369     1.599574   
                              07      8000.0     0.021349     2.086226   
                              08      8000.0     0.139799     2.983964   
                              09      8000.0     0.012609     1.810619   
                              10      8000.0     0.091662     1.984203   
                              11      8000.0     0.178579     5.393973   
                              12      8000.0     0.135284     2.260764   
                   max        01      8000.0     0.625050     0.081653   
                              02      8000.0     0.593938     0.080088   
                              03      8000.0     0.614813     0.085414   
                              04      8000.0     0.591072     0.083029   
                              05      8000.0     0.613068     0.084733   
                              06      8000.0     0.601139     0.087980   
                              07      8000.0     0.593704     0.084820   
                              08      8000.0     0.618332     0.085818   
                              09      8000.0     0.593066     0.082448   
                              10      8000.0     0.613302     0.089511   
                              11      8000.0     0.592615     0.087695   
                              12      8000.0     0.602501     0.086897   
                   mean       01      8000.0     0.251411     0.081668   
                              02      8000.0     0.247304     0.073219   
                              03      8000.0     0.246081     0.083013   
                              04      8000.0     0.248013     0.072828   
                              05      8000.0     0.251655     0.080661   
                              06      8000.0     0.243285     0.074201   
                              07      8000.0     0.243659     0.069708   
                              08      8000.0     0.251214     0.078578   
                              09      8000.0     0.250265     0.070979   
                              10      8000.0     0.242505     0.080035   
                              11      8000.0     0.231333     0.072339   
                              12      8000.0     0.229890     0.075132   
                   median     01      8000.0     0.241075     0.097057   
                              02      8000.0     0.239812     0.087417   
                              03      8000.0     0.234581     0.099444   
                              04      8000.0     0.240343     0.086907   
                              05      8000.0     0.241656     0.096697   
                              06      8000.0     0.233047     0.088789   
                              07      8000.0     0.235382     0.083456   
                              08      8000.0     0.240620     0.093835   
                              09      8000.0     0.243980     0.084434   
                              10      8000.0     0.229967     0.096487   
                              11      8000.0     0.220313     0.086823   
                              12      8000.0     0.217519     0.090408   
                   min        01      8000.0     0.007271     0.028201   
                              02      8000.0     0.007201     0.027493   
                              03      8000.0     0.006771     0.026500   
                              04      8000.0     0.007337     0.026097   
                              05      8000.

In [52]:
from IPython.display import display, Markdown
import pandas as pd

# Creating the structural classification data mapping as requested
classification_data = [
    {
        "Variable type": "Continuous numeric",
        "Examples": "`track/duration`, `features.csv` (all 518 float64 audio metrics like MFCC mean, chroma std, spectral centroid)",
        "Summaries": "Mean, Median, Standard Deviation, Min/Max, Percentiles, Skewness, Kurtosis",
        "Plots": "Histograms, Kernel Density Estimate (KDE) plots, Box plots, Violin plots"
    },
    {
        "Variable type": "Discrete numeric (counts)",
        "Examples": "`album/comments`, `album/favorites`, `artist/comments`, `track/favorites`, `track/interest`, `track/listens`",
        "Summaries": "Median, Mode, Range, Min/Max, Frequency distributions",
        "Plots": "Bar charts, Rootograms, Frequency polygons, Step plots"
    },
    {
        "Variable type": "Nominal categorical",
        "Examples": "`track/genre_top`, `track/license`, `album/type`, `track/language_code`",
        "Summaries": "Frequency counts, Mode, Percentages, Contingency tables (for relationships)",
        "Plots": "Bar plots (ordered by frequency), Pie charts, Countplots"
    },
    {
        "Variable type": "Ordinal categorical",
        "Examples": "`set/subset` (small < medium < large)",
        "Summaries": "Median, Mode, Cumulative frequencies, Percentages",
        "Plots": "Ordered bar charts, Cumulative distribution frequency line plots"
    },
    {
        "Variable type": "Binary",
        "Examples": "Dummy encoded features, or custom splits (e.g., `is_instrumental` from track genres/tags)",
        "Summaries": "Proportions, Mode, Success/Failure rates",
        "Plots": "Bar charts, Pie charts, Stacked bar charts"
    },
    {
        "Variable type": "Date/time",
        "Examples": "`album/date_created`, `album/date_released`, `artist/date_created`, `track/date_created`",
        "Summaries": "Minimum/Maximum date, Range, Seasonality/Trend decompositions",
        "Plots": "Time series line plots, Seasonal subseries plots, Calendar heatmaps"
    },
    {
        "Variable type": "Free text",
        "Examples": "`track/title`, `artist/bio`, `album/information`, `artist/name`, `track/tags`",
        "Summaries": "Word counts, Character lengths, Most frequent tokens/n-grams, Lexicon analysis",
        "Plots": "Word clouds, Bar charts of top terms, Text length histograms"
    },
    {
        "Variable type": "Identifier",
        "Examples": "`track_id` (Index), `album/id`, `artist/id`",
        "Summaries": "Uniqueness ratio, Total count, Cardinality",
        "Plots": "None (strictly used for join keys and record reference)"
    }
]

class_df = pd.DataFrame(classification_data)
markdown_table = class_df.to_markdown(index=False)

display(Markdown("### **Variable Classification Schema & Rules**\nThis framework establishes which statistical summaries and visualization plots are valid for each data category in the dataset:\n\n" + markdown_table))

### **Variable Classification Schema & Rules**
This framework establishes which statistical summaries and visualization plots are valid for each data category in the dataset:

| Variable type             | Examples                                                                                                       | Summaries                                                                      | Plots                                                                    |
|:--------------------------|:---------------------------------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------|:-------------------------------------------------------------------------|
| Continuous numeric        | `track/duration`, `features.csv` (all 518 float64 audio metrics like MFCC mean, chroma std, spectral centroid) | Mean, Median, Standard Deviation, Min/Max, Percentiles, Skewness, Kurtosis     | Histograms, Kernel Density Estimate (KDE) plots, Box plots, Violin plots |
| Discrete numeric (counts) | `album/comments`, `album/favorites`, `artist/comments`, `track/favorites`, `track/interest`, `track/listens`   | Median, Mode, Range, Min/Max, Frequency distributions                          | Bar charts, Rootograms, Frequency polygons, Step plots                   |
| Nominal categorical       | `track/genre_top`, `track/license`, `album/type`, `track/language_code`                                        | Frequency counts, Mode, Percentages, Contingency tables (for relationships)    | Bar plots (ordered by frequency), Pie charts, Countplots                 |
| Ordinal categorical       | `set/subset` (small < medium < large)                                                                          | Median, Mode, Cumulative frequencies, Percentages                              | Ordered bar charts, Cumulative distribution frequency line plots         |
| Binary                    | Dummy encoded features, or custom splits (e.g., `is_instrumental` from track genres/tags)                      | Proportions, Mode, Success/Failure rates                                       | Bar charts, Pie charts, Stacked bar charts                               |
| Date/time                 | `album/date_created`, `album/date_released`, `artist/date_created`, `track/date_created`                       | Minimum/Maximum date, Range, Seasonality/Trend decompositions                  | Time series line plots, Seasonal subseries plots, Calendar heatmaps      |
| Free text                 | `track/title`, `artist/bio`, `album/information`, `artist/name`, `track/tags`                                  | Word counts, Character lengths, Most frequent tokens/n-grams, Lexicon analysis | Word clouds, Bar charts of top terms, Text length histograms             |
| Identifier                | `track_id` (Index), `album/id`, `artist/id`                                                                    | Uniqueness ratio, Total count, Cardinality                                     | None (strictly used for join keys and record reference)                  |

####**Analysis of `tracks.csv` and `features.csv` Data Inspection**
  
Check that the row count and columns match what the source says.


Confirm each column was read with the right type: numbers stored as text, dates read as strings, and ID codes read as numbers are common errors.

# **III. Data quality assessment**

In [37]:
#tracks.csv

# Missing values: count and percent per column
print("--- Missing Values ---")
missing = small_tracks.isna().sum().to_frame("n_missing")
missing["pct"] = (missing["n_missing"] / len(small_tracks) * 100).round(1)
display(missing.sort_values("pct", ascending=False).head(20))

# Duplicate rows and duplicate IDs
print("\n--- Duplicate Rows and Duplicate IDs ---")
print(f"Duplicate rows in tracks: {small_tracks.duplicated().sum()}")
print(f"Duplicate track IDs in tracks (Index): {small_tracks.index.duplicated().sum()}")

# Categorical levels: look for typos, mixed case, stray spaces
print("\n--- Categorical Levels ---")
print("Unique values per column")
for col in small_tracks.select_dtypes("object").columns:
    print(col, small_tracks[col].nunique(), small_tracks[col].unique()[:5])

# Impossible values: check against what is physically or logically possible (e.g. non-positive duration)
print("\n--- Impossible Values ---")
impossible_durations = small_tracks[small_tracks[("track", "duration")] <= 0]
print(f"Tracks with non-positive duration: {len(impossible_durations)}")

# Displaying the 16 tracks with a duration of 0 or less
if len(impossible_durations) > 0:
    print("\n--- Impossible Duration Tracks ---")
    impossible_tracks_view = impossible_durations[[
        ('track', 'title'),
        ('artist', 'name'),
        ('track', 'duration'),
        ('set', 'subset')
    ]]
    display(impossible_tracks_view)

--- Missing Values ---


n_missing   pct
track  lyricist                7974  99.7
       publisher               7924  99.1
       information             7841  98.0
       composer                7820  97.8
artist active_year_end         7642  95.5
       wikipedia_page          7590  94.9
track  date_recorded           7535  94.2
artist related_projects        7306  91.3
       associated_labels       7151  89.4
track  language_code           6995  87.4
album  engineer                6858  85.7
       producer                6596  82.4
artist active_year_begin       6458  80.7
       longitude               5101  63.8
       latitude                5101  63.8
       members                 4475  55.9
album  date_released           2663  33.3
artist location                2529  31.6
       bio                     2086  26.1
       website                 1548  19.4


--- Duplicate Rows and Duplicate IDs ---
Duplicate rows in tracks: 0
Duplicate track IDs in tracks (Index): 0

--- Categorical Levels ---
Unique values per column
('album', 'date_created') 2459 ['2008-11-26 01:44:45' '2008-11-26 01:45:08' '2008-11-26 01:49:59'
 '2008-11-26 01:49:57' '2008-11-26 01:50:19']
('album', 'date_released') 1223 ['2009-01-05 00:00:00' '2008-02-06 00:00:00' '2007-05-22 00:00:00'
 '2009-01-16 00:00:00' '2009-01-15 00:00:00']
('album', 'engineer') 145 [nan 'self-recorded' 'Michael Goodstein' 'Irene Trudel' 'Scott Williams']
('album', 'information') 1942 ['<p></p>' nan
 '<p>Alec K. Redfearn &amp; The Eyesores: Ellen Santaniello (soprano); Domenick Panzarella (guitar); Steve Jobe (hurdy-gurdy); Sarah Tolan-Mee, Olivia Geiger (violin); Laura Gulley (viola); Alec K. Redfearn (accordion); Erica Schattle (bassoon); Jason <span id="switchNoteSnippet" style="display: none;">...&nbsp;&nbsp;&nbsp; <a class="notimportantlink" style="cursor: pointer;" onclick="javascript:sho

In [40]:
# features.csv

# 1. Missing values: count and percent per column
print("--- Missing Values in Features ---")
missing = small_features.isna().sum().to_frame("n_missing")
missing["pct"] = (missing["n_missing"] / len(small_features) * 100).round(1)
display(missing.sort_values("pct", ascending=False).head(20))

# 2. Duplicate rows and duplicate IDs
print("\n--- Duplicate Rows and Duplicate Track IDs in Features ---")
print(f"Duplicate rows in features: {small_features.duplicated().sum()}")
print(f"Duplicate track IDs (Index): {small_features.index.duplicated().sum()}")

# 3. Numeric boundaries: check for Infinite values, which can disrupt machine learning models
print("\n--- Infinite Values check ---")
inf_counts = np.isinf(small_features).sum().sum()
print(f"Total infinite (inf / -inf) values: {inf_counts}")

# 4. Check for invalid feature metrics (like negative standard deviations)
print("\n--- Statistical Sanity Check ---")
# Extracting columns that represent standard deviation
std_cols = [col for col in small_features.columns if col[1] == 'std']
invalid_stds = (small_features[std_cols] < 0).sum().sum()
print(f"Standard deviation columns with negative values: {invalid_stds}")

--- Missing Values in Features ---


n_missing  pct
feature     statistics number                
chroma_cens kurtosis   01              0  0.0
                       02              0  0.0
                       03              0  0.0
                       04              0  0.0
                       05              0  0.0
                       06              0  0.0
                       07              0  0.0
                       08              0  0.0
                       09              0  0.0
                       10              0  0.0
                       11              0  0.0
                       12              0  0.0
            max        01              0  0.0
                       02              0  0.0
                       03              0  0.0
                       04              0  0.0
                       05              0  0.0
                       06              0  0.0
                       07              0  0.0
                       08              0  0.0


--- Duplicate Rows and Duplicate Track IDs in Features ---
Duplicate rows in features: 17
Duplicate track IDs (Index): 0

--- Infinite Values check ---
Total infinite (inf / -inf) values: 0

--- Statistical Sanity Check ---
Standard deviation columns with negative values: 0


### **Summary of the Data quality assessment Results**

**1.   `tracks.csv`**

**Missing Values:**
*   There is an extremely high percenteage of missing values mostly on columns `('track', 'lyricists')`, `('track', 'publisher')`, `('track', 'information')`, and `('track', 'composer')` exceeding 90%. These can be safely ignored or dropped during cleaning.
*   `('track', 'genre_top')` is missing 53.5% of its values across the entire raw dataset. This is because many tracks are categorised only in sub-genres, and we will need to filter for tracks that actually have this label (such as the fma_small subset).


**Duplicates:**
*   There are no duplicate rows nor are there duplicate track IDs. The dataset has excellent row integrity with unique track identifiers.

**Impossible Values:**
*   There are 16 tracks with a duration of 0 or less, which is physically impossible for a song. These should be filtered out during the cleaning phase.


<br>

**2.   `tracks.csv`**

**Missing Values:**
*   There are absolutely no missing values (0.0% missing) across all 518 feature columns.

**Duplicates:**
*   There are 17 duplicated feature rows meaning 17 tracks have mathematically identical feature representations, which might indicated duplicated audio files or highly redundant silence/test tones in the small subset.

**Impossible Values:**
*   There are no infinite values and negative standard deviations present to disrupt the model confirming complete statistical sanity.


# **IV. Data cleaning**
*(with a log of every change)*

# **V. Univariate analysis**

# **VI. Bivariate and multivariate analysis**

# **VII. Type-specific analysis**
*(time, text, images, groups, as applies)*

# **VIII. Summary of findings**

# **IX. Limitations and next steps**

# **X. References**